# Reproduce the grading evaluation

Extract the upload package in a GPU notebook workspace. Set PACKAGE_ROOT below to the extracted directory. The saved results remain under data/ and results/. New generations are written to a separate directory.

The model weights are downloaded from the immutable revision recorded in settings.json. The recorded study used a Tesla T4, NF4 and float16 compute; outputs may differ on other software or hardware. New outputs require new human reviews.


In [ ]:
from pathlib import Path
import subprocess
import sys

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "generate.py").exists() and (PACKAGE_ROOT.parent / "generate.py").exists():
    PACKAGE_ROOT = PACKAGE_ROOT.parent
if not (PACKAGE_ROOT / "generate.py").exists():
    raise FileNotFoundError("Set PACKAGE_ROOT to the extracted upload package.")
print(PACKAGE_ROOT)


## Check the saved results on CPU

No model downloads or additional packages are needed for this cell.


In [ ]:
subprocess.run([sys.executable, "-B", str(PACKAGE_ROOT / "analyze.py"), "--check"], check=True)
subprocess.run([sys.executable, "-B", str(PACKAGE_ROOT / "generate.py"), "--blocks", "all"], check=True)


## Install the GPU dependencies

Run this cell in an NVIDIA GPU runtime when you want to produce new responses. The recorded Torch/CUDA versions are in settings.json. Restart the runtime after installation if required.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PACKAGE_ROOT / "requirements-gpu.txt")], check=True)


## Generate all six blocks

This cell makes 348 model calls. Use a fresh output directory each time. Replace all with greedy for a 58-response reference run.


In [ ]:
OUTPUT_DIRECTORY = PACKAGE_ROOT.parent / "grading_regenerated"
subprocess.run([sys.executable, "-B", str(PACKAGE_ROOT / "generate.py"), "--blocks", "all", "--generate", "--output", str(OUTPUT_DIRECTORY)], check=True)
